## Directories

In [ ]:
info_dir = '/path/to/_info'
ihc_res_dir = '/path/to/ihc/_res'

## Data Preprocessing

### Load, Drop Columns and Check Non-Values

In [ ]:
import pandas as pd
import numpy as np

BIOMARKERS = []
NAN_VALS = []
TO_DROP = []

sample_data = pd.read_csv(f"{info_dir}/_data/sample_data.csv")
sample_data = sample_data.drop(columns=TO_DROP, errors="ignore")

print(">>> Check Non-valid Values for Sample Cohort")
for col in sample_data.columns:

    non_valid = []
    if col=='id' or col=='subtype':
        continue

    for value in sample_data[col]:

        try:
            _ = float(value)
        except ValueError:
            non_valid.append(value)

    if non_valid:
        print(f"    - Column '{col}' Contains Non-valid Values: {non_valid}")

columns_to_convert = [col for col in sample_data.columns if col not in ['id', 'subtype']]
sample_data[columns_to_convert] = sample_data[columns_to_convert].apply(pd.to_numeric, errors='coerce')

p_cols = [col for col in sample_data.columns if col.endswith('_p')]
for p_col in p_cols:
    base = p_col[:-2] 
    i_col = f"{base}_i"
    
    if i_col in sample_data.columns:
        q_col = f"{base}"
        sample_data[q_col] = sample_data[p_col] * sample_data[i_col]

sample_data = sample_data.drop(columns=p_cols + [f"{col[:-2]}_i" for col in p_cols])

In [ ]:
print(">>> Sample Cohort:")
print(sample_data['subtype'].value_counts())

### Apply PCA to See Distribution of Training Data

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px

X = sample_data.drop(columns=["id", "subtype"])
y = sample_data["subtype"]
sample_ids = sample_data["id"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(data=X_pca, columns=["PC1", "PC2", "PC3"])
pca_df["subtype"] = y.values
pca_df["id"] = sample_ids

fig = px.scatter_3d(
    pca_df,
    x="PC1", y="PC2", z="PC3",
    color="subtype",
    hover_name="id",
    height=700
)

fig.update_traces(marker=dict(size=6))
fig.show()

## mRMR Feature Selection

#### Apply mRMR For Feature Selection

In [ ]:
from sklearn.preprocessing import LabelEncoder

X_mrmr = sample_data.drop(columns=['id', 'subtype'])

label_encoder = LabelEncoder()
y_mrmr = label_encoder.fit_transform(sample_data['subtype'])
subtypes = label_encoder.classes_

#### Sort the Most Important Features

In [ ]:
from mrmr import mrmr_classif

(selected_features, relevance, redundancy) = mrmr_classif(X=X_mrmr, y=y_mrmr, K=len(BIOMARKERS), return_scores=True)

#### Plot Redundancy

In [ ]:
red_df = redundancy.copy()

row_sums = red_df.sum(axis=1)
col_sums = red_df.sum(axis=0)
ordered_red = red_df.loc[row_sums.sort_values(ascending=False).index, col_sums.sort_values(ascending=False).index]

rev_red = ordered_red.iloc[::-1]

mask = np.triu(np.ones_like(rev_red, dtype=bool))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

white_red = LinearSegmentedColormap.from_list("white_red", ["white", "red"])

plt.figure(figsize=(8, 6))

ax = sns.heatmap(
    rev_red,
    mask=mask,
    annot=True,
    cmap=white_red,
    annot_kws={"size": 16, "color": "black"},
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Redundancy Score'}
)

ax.set_xticklabels(ax.get_xticklabels(), fontsize=16, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=16, rotation=0)

colorbar = ax.collections[0].colorbar
colorbar.ax.tick_params(labelsize=14)
colorbar.set_label("Redundancy Score", fontsize=18, labelpad=10)

plt.tight_layout()
plt.show()

#### Plot Relevancy

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df = relevance.reset_index()
df.columns = ['Marker', 'Relevancy']
df['Marker'] = df['Marker'].astype(str)

order = df.sort_values('Relevancy', ascending=False)['Marker'].tolist()
palette = sns.color_palette("Greens", n_colors=len(df))[::-1]

plt.figure(figsize=(6, 6))
sns.barplot(
    data=df,
    x='Marker',
    y='Relevancy',
    palette=palette,
    order=order
)

plt.xlabel('IHC Biomarker', fontsize=18)
plt.ylabel('Relevancy', fontsize=18)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.show()

#### Mutual Info Matrix

In [ ]:
from sklearn.feature_selection import mutual_info_classif

RANDOM_SEED = 1

mi_matrix = pd.DataFrame(index=X_mrmr.columns, columns=subtypes)

for i, subtype in enumerate(subtypes):
    y_binary = (y_mrmr == i).astype(int)
    mi_scores = mutual_info_classif(X_mrmr, y_binary, discrete_features=False, random_state=RANDOM_SEED)
    mi_matrix[subtype] = mi_scores

# Step 3: Optional - Normalize or sort
mi_matrix = mi_matrix.astype(float)
mi_matrix_sorted = mi_matrix.sort_values(by=subtypes.tolist(), ascending=False)

mi_matrix_sorted

## IHC Classifier Training

#### Load Data Split IDs

In [ ]:
import json
with open(f'{info_dir}/split_ids.json', 'r') as f:
    split_ids = json.load(f)

In [ ]:
train_dfs = {}
val_dfs = {}

for fold_name, fold_data in split_ids.items():
    
    train_ids = {subtype: set(ids) for subtype, ids in fold_data['Train'].items()}
    val_ids = {subtype: set(ids) for subtype, ids in fold_data['Test'].items()}
    
    train_mask = sample_data.apply(lambda row: row['id'] in train_ids.get(row['subtype'], set()), axis=1)
    val_mask = sample_data.apply(lambda row: row['id'] in val_ids.get(row['subtype'], set()), axis=1)
    
    train_dfs[fold_name] = sample_data[train_mask].reset_index(drop=True)
    val_dfs[fold_name] = sample_data[val_mask].reset_index(drop=True)

#### Train the Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import ParameterGrid
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, recall_score, confusion_matrix

RANDOM_SEED = 1

def compute_specificity(y_true, y_pred, average='macro'):
    labels = np.unique(np.concatenate([y_true, y_pred]))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tn_fp = cm.sum(axis=0) - np.diag(cm)
    fn_tp = cm.sum(axis=1) - np.diag(cm)
    tn = cm.sum() - (tn_fp + fn_tp + np.diag(cm))
    fp = tn_fp
    specificity_per_class = tn / (tn + fp + 1e-10)
    if average == 'macro':
        return np.mean(specificity_per_class)
    else:
        return specificity_per_class

model_grids = {
    'DecisionTree': (DecisionTreeClassifier, {
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5, 10],
        'criterion': ['gini', 'entropy']
    }),
    'RandomForest': (RandomForestClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }),
    'XGBoost': (XGBClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1]
    }),
    'ExtraTrees': (ExtraTreesClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [None, 5, 10],
        'min_samples_split': [2, 5]
    }),
    'GradientBoosting': (GradientBoostingClassifier, {
        'n_estimators': [100, 200],
        'max_depth': [3, 5],
        'learning_rate': [0.01, 0.1]
    }),
    'SVM': (SVC, {
        'C': [0.1, 1, 10],
        'kernel': ['linear', 'rbf']
    }),
    'LogisticRegression': (LogisticRegression, {
        'C': [0.1, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear', 'saga'],
        'max_iter': [100, 200]
    }),
    'KNN': (KNeighborsClassifier, {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    })
}

results = []
total_iterations = 0
for model_name, (ModelClass, param_grid) in model_grids.items():
    param_combinations = list(ParameterGrid(param_grid))
    total_iterations += len(param_combinations) * len(train_dfs)

progress_bar = tqdm(total=total_iterations, desc=">>> Model Training", position=0, leave=True)
for model_name, (ModelClass, param_grid) in model_grids.items():
    
    param_combinations = list(ParameterGrid(param_grid))
    for params in param_combinations:
        
        val_scores = []
        f1_scores = []
        recall_scores = []
        specificity_scores = []
        models = []
        misclassified_ids = {}
        
        for fold_name in train_dfs.keys():
            
            train_data = train_dfs[fold_name].sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
            X_train = train_data.drop(columns=['subtype', 'id'])
            y_train = train_data['subtype']
            
            X_val = val_dfs[fold_name].drop(columns=['subtype', 'id'])
            y_val = val_dfs[fold_name]['subtype']
            val_ids = val_dfs[fold_name]['id']
            
            label_encoder = LabelEncoder()
            y_train = label_encoder.fit_transform(y_train)
            y_val = label_encoder.transform(y_val)
            
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)
            
            try:
                if 'random_state' in ModelClass().get_params():
                    model = ModelClass(**params, random_state=RANDOM_SEED)
                else:
                    model = ModelClass(**params)
                    
                model.fit(X_train, y_train)
                
                y_pred = model.predict(X_val)
                val_scores.append(accuracy_score(y_val, y_pred))
                f1_scores.append(f1_score(y_val, y_pred, average='macro'))
                recall_scores.append(recall_score(y_val, y_pred, average='macro'))
                specificity_scores.append(compute_specificity(y_val, y_pred, average='macro'))
                models.append(model)
                
                misclassified_mask = (y_pred != y_val)
                misclassified_ids[fold_name] = val_ids[misclassified_mask].tolist()
            
            except Exception:
                pass
            
            progress_bar.update(1)

        if len(val_scores) > 0:

            avg_val_score = np.mean(val_scores)
            avg_f1_score = np.mean(f1_scores)
            avg_recall_score = np.mean(recall_scores)
            avg_specificity_score = np.mean(specificity_scores)
            results.append({
                'model': model_name,
                'params': params,
                'avg_val_score': avg_val_score,
                'avg_f1_score': avg_f1_score,
                'avg_recall_score': avg_recall_score,
                'avg_specificity_score': avg_specificity_score,
                'val_scores': val_scores,
                'f1_scores': f1_scores,
                'recall_scores': recall_scores,
                'specificity_scores': specificity_scores,
                'models': models,
                'miss_ids': misclassified_ids
            })

progress_bar.close()
results_df = pd.DataFrame(results).sort_values(by='avg_val_score', ascending=False)
results_df.to_csv(f"{ihc_res_dir}/grid_search.csv", index=False)
best_per_model = results_df.loc[results_df.groupby('model')['avg_val_score'].idxmax()]

#### Get the Best Model (Biomarkers Sorted By mRMR)

In [ ]:
results_by_biomarker_count = []
best_entry = results_df.iloc[0]

best_model = best_entry['model']
best_params = best_entry['params']

for i in range(1, len(BIOMARKERS) + 1):

    selected = BIOMARKERS[:i]

    full_train_df = pd.concat(train_dfs.values()).sample(frac=1, random_state=RANDOM_SEED)
    X_train_full = full_train_df[selected]
    y_train_full = full_train_df['subtype']
    train_ids = full_train_df['id']

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train_full)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_full)

    ModelClass = model_grids[best_model][0]
    if 'random_state' in ModelClass().get_params():
        model = ModelClass(**best_params, random_state=RANDOM_SEED)
    else:
        model = ModelClass(**best_params)

    model.fit(X_train_scaled, y_train_encoded)
    y_pred_train = model.predict(X_train_scaled)
    train_accuracy = accuracy_score(y_train_encoded, y_pred_train)

    misclassified_train_ids = train_ids[(y_pred_train != y_train_encoded)].tolist()
    results_by_biomarker_count.append({
        'n_biomarkers': i,
        'biomarkers': selected,
        'samnple_acc': train_accuracy,
        'miss_sample_ids': misclassified_train_ids
    })

results_df_biomarkers = pd.DataFrame(results_by_biomarker_count)